# Khukuri Quick Start

**5-minute introduction to Khukuri AMR drug discovery platform**

This notebook will:
1. Set up your OpenAI API key
2. Download resistance databases
3. Run a simple AMR analysis
4. Show basic docking workflow

---

## Cell 1: Setup OpenAI API Key

**Required for AI agents to work**

In [ ]:
import os
import getpass

# Check if API key already set
if not os.environ.get('OPENAI_API_KEY'):
    print("OpenAI API key not found in environment.")
    print("Get your API key from: https://platform.openai.com/api-keys\n")
    
    api_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ['OPENAI_API_KEY'] = api_key
    print("✓ API key set successfully")
else:
    print("✓ OpenAI API key already configured")

# Verify key works
try:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    # Test with minimal call
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "test"}],
        max_tokens=5
    )
    print("✓ API key verified and working")
except Exception as e:
    print(f"✗ API key verification failed: {e}")
    print("Please check your API key and try again")

## Cell 2: Install Dependencies & Import Modules

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent))

# Import Khukuri modules
from src.bioknowledge import ResistanceDatabase, PathogenDatabase, CARDDownloader
from src.docking import StructureDownloader
from src.genomics import ResistanceGenomicsAnalyzer
from src.microbiology import MICAnalyzer
from src.core import setup_logger

# Setup logging
logger = setup_logger('khukuri')

print("✓ All modules imported successfully")
print("✓ Khukuri v2.0 ready")

## Cell 3: Download Resistance Database

**Downloads CARD database or uses curated fallback**

In [ ]:
print("Downloading CARD resistance database...\n")

# Initialize downloader
card = CARDDownloader()

# Try to download CARD
success = card.download_card()

# Update resistance database
db_file = card.update_resistance_db_file()

if success:
    print("\n✓ CARD database downloaded successfully")
else:
    print("\n⚠ CARD download failed, using curated fallback database")
    print("  (Contains 22 essential AMR genes for TB, S. aureus, E. coli)")

print(f"\n✓ Database saved to: {db_file}")

# Load and verify
resistance_db = ResistanceDatabase()
print(f"\n✓ Loaded {len(resistance_db.genes)} resistance genes")
print(f"✓ Loaded {len(resistance_db.mechanisms)} resistance mechanisms")

## Cell 4: Explore Resistance Database

In [ ]:
# Query TB resistance genes
tb_genes = resistance_db.get_genes_by_organism('tuberculosis')
print(f"TB Resistance Genes ({len(tb_genes)}):")
for gene in tb_genes:
    info = resistance_db.query_gene(gene)
    print(f"  • {gene}: {info['type']} resistance via {info['mechanism']}")

print("\n" + "="*60 + "\n")

# Query S. aureus resistance genes
staph_genes = resistance_db.get_genes_by_organism('aureus')
print(f"S. aureus Resistance Genes ({len(staph_genes)}):")
for gene in staph_genes:
    info = resistance_db.query_gene(gene)
    print(f"  • {gene}: {info['type']} resistance via {info['mechanism']}")

print("\n" + "="*60 + "\n")

# Show resistance mechanisms
print("Resistance Mechanisms:")
for mechanism, examples in resistance_db.mechanisms.items():
    print(f"  • {mechanism}: {', '.join(examples[:2])}")

## Cell 5: Analyze Resistance Mutations

In [ ]:
# Initialize genomics analyzer
analyzer = ResistanceGenomicsAnalyzer()

# Analyze rifampicin resistance mutations in rpoB
print("Analyzing rifampicin resistance mutations in rpoB gene:\n")

mutations = ['S531L', 'H526Y', 'D516V']
profile = analyzer.analyze_mutation_profile('rpoB', mutations)

print(f"Gene: {profile['gene']}")
print(f"Mutations analyzed: {len(profile['mutations'])}")
print(f"Predicted resistance: {profile['predicted_resistance']}")
print(f"Confidence: {profile['confidence']:.2f}")

print("\nMutation details:")
for mut in profile['mutations']:
    print(f"  • {mut['mutation']}: {mut['effect']} (impact: {mut['impact']})")

print("\n" + "="*60 + "\n")

# Analyze isoniazid resistance
print("Analyzing isoniazid resistance mutations in katG gene:\n")

profile2 = analyzer.analyze_mutation_profile('katG', ['S315T'])
print(f"Gene: {profile2['gene']}")
print(f"Predicted resistance: {profile2['predicted_resistance']}")
print(f"Confidence: {profile2['confidence']:.2f}")

## Cell 6: Download Protein Structures

In [ ]:
# Initialize structure downloader
downloader = StructureDownloader()

print("Downloading TB drug target structures...\n")

# Define targets
targets = [
    {'name': 'InhA', 'pdb_id': '1P44', 'organism': 'Mycobacterium tuberculosis'},
    {'name': 'KatG', 'pdb_id': '2CCA', 'organism': 'Mycobacterium tuberculosis'}
]

# Download structures
structures = downloader.download_batch(targets)

print(f"\n✓ Downloaded {len(structures)}/{len(targets)} structures:\n")
for target_name, path in structures.items():
    file_size = path.stat().st_size / 1024  # KB
    print(f"  • {target_name}: {path.name} ({file_size:.1f} KB)")

# Show all available structures
available = downloader.get_available_structures()
print(f"\n✓ Total structures available: {len(available)}")

## Cell 7: MIC Analysis Example

In [ ]:
# Initialize MIC analyzer
mic_analyzer = MICAnalyzer()

print("Adding MIC test results...\n")

# Add test results for rifampicin
mic_analyzer.add_mic_result('COMP_001', 'ATCC_25923', 'rifampicin', 0.5)
mic_analyzer.add_mic_result('COMP_002', 'ATCC_25923', 'rifampicin', 2.0)
mic_analyzer.add_mic_result('COMP_003', 'ATCC_25923', 'rifampicin', 0.25)

# Add test results for isoniazid
mic_analyzer.add_mic_result('COMP_001', 'ATCC_25923', 'isoniazid', 0.1)
mic_analyzer.add_mic_result('COMP_002', 'ATCC_25923', 'isoniazid', 0.5)

print(f"✓ Added {len(mic_analyzer.mic_data)} MIC results\n")

# Get strain profile
profile = mic_analyzer.get_mic_profile('ATCC_25923')

print("Strain Profile (ATCC_25923):")
print(f"  Total tests: {profile['total_tests']}")
print(f"  Susceptible: {profile['susceptible_count']}")
print(f"  Intermediate: {profile['intermediate_count']}")
print(f"  Resistant: {profile['resistance_count']}")

print("\nMIC Values by Drug:")
for drug, mic_value in profile['mic_values'].items():
    print(f"  • {drug}: {mic_value} μg/mL")

## Cell 8: Summary & Next Steps

In [ ]:
print("="*60)
print("QUICK START COMPLETE")
print("="*60)

print("\n✓ What you've done:")
print("  1. Configured OpenAI API for AI agents")
print("  2. Downloaded resistance database (22+ genes)")
print("  3. Analyzed resistance mutations")
print("  4. Downloaded protein structures")
print("  5. Performed MIC analysis")

print("\n📚 Next Steps:")
print("  • Run 04_amr_discovery.ipynb for full AI-driven workflow")
print("  • See examples/amr_discovery_example.py for Python script")
print("  • Read docs/amr_features.md for detailed documentation")

print("\n🔬 System Status:")
print(f"  • Resistance genes: {len(resistance_db.genes)}")
print(f"  • Protein structures: {len(available)}")
print(f"  • MIC results: {len(mic_analyzer.mic_data)}")
print(f"  • AI agents: Ready (OpenAI API configured)")

print("\n" + "="*60)